# OpenVals — AI Model Validation from Zero
## Episode 02: How Do We Know an AI Model Actually Works?

A model score is not automatically evidence of trustworthiness.

This notebook studies classification performance through exact definitions, direct calculations, controlled experiments, and statistical simulation.

### Core principle

> **A metric is meaningful only when its mathematical definition, data distribution, decision context, and uncertainty are understood.**

This episode covers classification outcomes, confusion matrices, accuracy, precision, recall, specificity, false-positive and false-negative rates, F1 score, class imbalance, balanced accuracy, sample size, sampling variability, confidence intervals, Monte Carlo simulation, and cost-sensitive evaluation.


# 1. Binary Classification

Let the true class be

\[
Y\in\{0,1\}
\]

and the predicted class be

\[
\hat{Y}\in\{0,1\}.
\]

The four possible outcomes are

\[
TP=\#\{Y=1,\hat{Y}=1\}
\]

\[
FN=\#\{Y=1,\hat{Y}=0\}
\]

\[
FP=\#\{Y=0,\hat{Y}=1\}
\]

\[
TN=\#\{Y=0,\hat{Y}=0\}.
\]

The total number of evaluated observations is

\[
N=TP+TN+FP+FN.
\]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt

print("NumPy:", np.__version__)
print("Environment ready.")


# 2. The 95% Accuracy Example

Consider

\[
N=1000
\]

observations, with

\[
950
\]

negative cases and

\[
50
\]

positive cases.

A classifier predicts the negative class for every observation.

Therefore,

\[
TP=0,\qquad FN=50,\qquad FP=0,\qquad TN=950.
\]

Accuracy is

\[
\mathrm{Accuracy}
=
\frac{TP+TN}{TP+TN+FP+FN}.
\]

Substituting the values,

\[
\mathrm{Accuracy}
=
\frac{0+950}{1000}
=
0.95.
\]

Hence,

\[
\boxed{\mathrm{Accuracy}=95\%}.
\]


In [ ]:
TP, FN, FP, TN = 0, 50, 0, 950
N = TP + TN + FP + FN
accuracy = (TP + TN) / N

print("N        =", N)
print("TP       =", TP)
print("FN       =", FN)
print("FP       =", FP)
print("TN       =", TN)
print("Accuracy =", f"{accuracy:.2%}")


# 3. Recall Reveals a Different Property

Recall, also called sensitivity or true positive rate, is

\[
\mathrm{Recall}
=
\frac{TP}{TP+FN}.
\]

For this classifier,

\[
\mathrm{Recall}
=
\frac{0}{0+50}
=
0.
\]

Therefore,

\[
\boxed{\mathrm{Recall}=0\%}.
\]

The same classifier has

\[
\mathrm{Accuracy}=95\%
\]

and

\[
\mathrm{Recall}=0\%.
\]

There is no contradiction. These quantities measure different aspects of performance.


In [ ]:
recall = TP / (TP + FN)
print("Recall =", f"{recall:.2%}")


# 4. Confusion Matrix

The binary confusion matrix is

\[
\begin{array}{c|cc}
 & \hat{Y}=1 & \hat{Y}=0 \\
\hline
Y=1 & TP & FN \\
Y=0 & FP & TN
\end{array}
\]

For the current classifier,

\[
\begin{array}{c|cc}
 & \hat{Y}=1 & \hat{Y}=0 \\
\hline
Y=1 & 0 & 50 \\
Y=0 & 0 & 950
\end{array}
\]


In [ ]:
cm = np.array([[TP, FN],
               [FP, TN]])

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm)
ax.set_xticks([0, 1], labels=["Predicted Positive", "Predicted Negative"])
ax.set_yticks([0, 1], labels=["Actual Positive", "Actual Negative"])

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")

ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()


# 5. Core Classification Metrics

### Accuracy

\[
\mathrm{Accuracy}
=
\frac{TP+TN}{TP+TN+FP+FN}
\]

### Precision

\[
\mathrm{Precision}
=
\frac{TP}{TP+FP}
\]

If

\[
TP+FP=0,
\]

precision is mathematically undefined because the denominator is zero.

### Recall / Sensitivity / True Positive Rate

\[
\mathrm{Recall}
=
\mathrm{Sensitivity}
=
\mathrm{TPR}
=
\frac{TP}{TP+FN}
\]

### Specificity / True Negative Rate

\[
\mathrm{Specificity}
=
\mathrm{TNR}
=
\frac{TN}{TN+FP}
\]

### False Positive Rate

\[
\mathrm{FPR}
=
\frac{FP}{FP+TN}
=
1-\mathrm{Specificity}
\]

### False Negative Rate

\[
\mathrm{FNR}
=
\frac{FN}{FN+TP}
=
1-\mathrm{Recall}
\]

### F1 Score

\[
F_1
=
\frac{2\,\mathrm{Precision}\,\mathrm{Recall}}
{\mathrm{Precision}+\mathrm{Recall}}
\]

If the denominator is zero, the expression is undefined unless an explicit convention is adopted.


In [ ]:
def safe_divide(a, b):
    return np.nan if b == 0 else a / b

def classification_metrics(tp, tn, fp, fn):
    accuracy = safe_divide(tp + tn, tp + tn + fp + fn)
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    fpr = safe_divide(fp, fp + tn)
    fnr = safe_divide(fn, fn + tp)

    if np.isnan(precision) or np.isnan(recall) or precision + recall == 0:
        f1 = np.nan
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "FPR": fpr,
        "FNR": fnr,
        "F1": f1,
    }

metrics = classification_metrics(TP, TN, FP, FN)

for name, value in metrics.items():
    text = "undefined" if np.isnan(value) else f"{value:.2%}"
    print(f"{name:12s} = {text}")


# 6. Class Imbalance

Let

\[
\pi_0=\frac{N_0}{N}
\]

be the prevalence of the negative class and

\[
\pi_1=\frac{N_1}{N}
\]

the prevalence of the positive class.

A classifier that always predicts the majority class achieves

\[
\mathrm{Accuracy}_{\mathrm{majority}}
=
\max(\pi_0,\pi_1).
\]

Thus, high raw accuracy can occur even when the classifier has no useful ability to identify the minority class.


In [ ]:
distributions = [(950,50),(800,200),(500,500),(990,10),(999,1)]

print("NEGATIVE  POSITIVE   ACCURACY   RECALL")
print("-" * 43)

rows = []
for negatives, positives in distributions:
    tp, fn, fp, tn = 0, positives, 0, negatives
    total = negatives + positives
    acc = (tp + tn) / total
    rec = tp / (tp + fn)
    rows.append((negatives, positives, acc, rec))
    print(f"{negatives:8d}  {positives:8d}   {acc:8.2%}   {rec:6.2%}")


In [ ]:
labels = [f"{n}/{p}" for n,p,_,_ in rows]
values = [a for _,_,a,_ in rows]

plt.figure(figsize=(8,5))
plt.bar(labels, values)
plt.ylim(0, 1.05)
plt.xlabel("Negative / Positive observations")
plt.ylabel("Accuracy")
plt.title("Accuracy of an Always-Negative Classifier")
plt.grid(axis="y", alpha=0.25)
plt.show()


# 7. Same Accuracy, Different Behaviour

Consider a balanced dataset of

\[
N=1000
\]

with

\[
500
\]

positive and

\[
500
\]

negative observations.

For Model A,

\[
TP=450,\quad FN=50,\quad FP=50,\quad TN=450.
\]

For Model B,

\[
TP=400,\quad FN=100,\quad FP=0,\quad TN=500.
\]

Both have

\[
\mathrm{Accuracy}
=
\frac{900}{1000}
=
0.90.
\]

Therefore,

\[
\boxed{
\mathrm{Accuracy}_A
=
\mathrm{Accuracy}_B
=
90\%
}
\]

while their recall, specificity, false-positive rate, and false-negative rate differ.


In [ ]:
model_A = classification_metrics(tp=450, tn=450, fp=50, fn=50)
model_B = classification_metrics(tp=400, tn=500, fp=0, fn=100)

print("METRIC         MODEL A     MODEL B")
print("-" * 36)

for metric in model_A:
    a, b = model_A[metric], model_B[metric]
    a = "undefined" if np.isnan(a) else f"{a:.2%}"
    b = "undefined" if np.isnan(b) else f"{b:.2%}"
    print(f"{metric:12s}   {a:>8s}   {b:>8s}")


# 8. Balanced Accuracy

Balanced accuracy for binary classification is

\[
\mathrm{Balanced\ Accuracy}
=
\frac{\mathrm{Sensitivity}+\mathrm{Specificity}}{2}.
\]

For the always-negative classifier,

\[
\mathrm{Sensitivity}=0
\]

and

\[
\mathrm{Specificity}=1.
\]

Hence,

\[
\mathrm{Balanced\ Accuracy}
=
\frac{0+1}{2}
=
0.5.
\]

Thus,

\[
\boxed{\mathrm{Balanced\ Accuracy}=50\%}
\]

while raw accuracy is

\[
95\%.
\]


In [ ]:
specificity = TN / (TN + FP)
balanced_accuracy = (recall + specificity) / 2

print("Raw accuracy      =", f"{accuracy:.2%}")
print("Balanced accuracy =", f"{balanced_accuracy:.2%}")


# 9. Sample Size and Sampling Uncertainty

Assume that each evaluated prediction is an independent Bernoulli trial with probability

\[
p
\]

of being correct.

Define

\[
X_i=
\begin{cases}
1,& \text{prediction }i\text{ is correct},\\
0,& \text{otherwise}.
\end{cases}
\]

Then

\[
X_i\sim\mathrm{Bernoulli}(p).
\]

Observed accuracy is

\[
\hat p
=
\frac{1}{n}\sum_{i=1}^{n}X_i.
\]

Under these assumptions,

\[
\mathbb{E}[\hat p]=p
\]

and

\[
\mathrm{Var}(\hat p)
=
\frac{p(1-p)}{n}.
\]

Therefore,

\[
\mathrm{SE}(\hat p)
=
\sqrt{\frac{p(1-p)}{n}}.
\]

Sampling uncertainty decreases proportionally to

\[
\frac{1}{\sqrt{n}}.
\]


# 10. Identical Point Estimates, Different Uncertainty

Consider

\[
\frac{9}{10}=0.90
\]

and

\[
\frac{900000}{1000000}=0.90.
\]

Both give the same point estimate,

\[
\hat p=0.90.
\]

Using the plug-in estimate of the standard error,

\[
\widehat{\mathrm{SE}}
=
\sqrt{\frac{\hat p(1-\hat p)}{n}},
\]

the estimated sampling uncertainty differs substantially because the sample sizes differ.


In [ ]:
def se_proportion(p_hat, n):
    return sqrt(p_hat * (1 - p_hat) / n)

examples = [(9,10),(90,100),(900,1000),(900000,1000000)]

print("CORRECT / N       p-hat        Estimated SE")
print("-" * 46)
for correct, n in examples:
    p_hat = correct / n
    se = se_proportion(p_hat, n)
    print(f"{correct:>7}/{n:<10} {p_hat:8.4f}      {se:.6f}")


# 11. Wilson Score Confidence Interval

A confidence interval describes uncertainty associated with estimating an unknown population proportion from a finite sample.

The elementary Wald interval is

\[
\hat p
\pm
z_{\alpha/2}
\sqrt{
\frac{\hat p(1-\hat p)}{n}
},
\]

but it can perform poorly for small samples and for proportions near zero or one.

The Wilson score interval is generally more reliable.

Let

\[
z=z_{1-\alpha/2}.
\]

Its center is

\[
c=
\frac{
\hat p+\frac{z^2}{2n}
}{
1+\frac{z^2}{n}
}
\]

and its half-width is

\[
h=
\frac{
z
}{
1+\frac{z^2}{n}
}
\sqrt{
\frac{\hat p(1-\hat p)}{n}
+
\frac{z^2}{4n^2}
}.
\]

The interval is

\[
[c-h,\;c+h].
\]


In [ ]:
def wilson_interval(successes, n, z=1.959963984540054):
    p_hat = successes / n
    denominator = 1 + z**2 / n
    center = (p_hat + z**2 / (2*n)) / denominator
    half_width = (
        z / denominator
        * sqrt(
            p_hat * (1-p_hat) / n
            + z**2 / (4*n**2)
        )
    )
    return center - half_width, center + half_width

print("OBSERVED        ACCURACY       95% WILSON INTERVAL")
print("-" * 57)

for successes, n in examples:
    p_hat = successes / n
    low, high = wilson_interval(successes, n)
    print(
        f"{successes:>7}/{n:<10} "
        f"{p_hat:8.2%}       "
        f"[{low:.2%}, {high:.2%}]"
    )


# 12. Monte Carlo Simulation

Assume the true probability of a correct prediction is

\[
p=0.90.
\]

For an evaluation sample of size

\[
n,
\]

the number of correct predictions follows

\[
K\sim\mathrm{Binomial}(n,p).
\]

Observed accuracy is

\[
\hat p=\frac{K}{n}.
\]

Repeated simulation approximates the sampling distribution of

\[
\hat p.
\]


In [ ]:
rng = np.random.default_rng(42)
true_p = 0.90
repetitions = 10000
sample_sizes = [10, 100, 1000]

simulation_results = {}

for n in sample_sizes:
    correct = rng.binomial(n=n, p=true_p, size=repetitions)
    observed = correct / n
    simulation_results[n] = observed

    print(
        f"n={n:4d} | "
        f"mean={observed.mean():.4f} | "
        f"sample SD={observed.std(ddof=1):.4f}"
    )


In [ ]:
for n in sample_sizes:
    values = simulation_results[n]

    plt.figure(figsize=(8,5))
    plt.hist(values, bins=30, edgecolor="black")
    plt.axvline(true_p, linestyle="--", linewidth=2)
    plt.xlabel("Observed Accuracy")
    plt.ylabel("Frequency")
    plt.title(f"Sampling Distribution of Accuracy, n={n}")
    plt.grid(alpha=0.2)
    plt.show()


Under the Bernoulli assumptions,

\[
\mathrm{SD}(\hat p)
=
\sqrt{
\frac{p(1-p)}{n}
}.
\]

For

\[
p=0.90,
\]

the theoretical standard deviations are

\[
n=10:
\qquad
\sqrt{\frac{0.9(0.1)}{10}}
\approx 0.09487
\]

\[
n=100:
\qquad
\sqrt{\frac{0.9(0.1)}{100}}
=0.03
\]

\[
n=1000:
\qquad
\sqrt{\frac{0.9(0.1)}{1000}}
\approx 0.00949.
\]


In [ ]:
for n in sample_sizes:
    theoretical = sqrt(true_p * (1-true_p) / n)
    empirical = simulation_results[n].std(ddof=1)
    print(
        f"n={n:4d} | "
        f"Theoretical SD={theoretical:.5f} | "
        f"Simulation SD={empirical:.5f}"
    )


# 13. Error Costs

Accuracy implicitly counts each correct classification equally and does not encode different consequences for different error types.

A simple cost-sensitive loss is

\[
L
=
c_{FP}FP+c_{FN}FN,
\]

where

\[
c_{FP}
\]

is the assigned cost of a false positive and

\[
c_{FN}
\]

is the assigned cost of a false negative.

If

\[
c_{FN}\gg c_{FP},
\]

a classifier with fewer false negatives can be preferable even if its raw accuracy is lower.

The cost values must be justified by the application and cannot be inferred from the classification metric alone.


In [ ]:
def weighted_error(fp, fn, cost_fp=1.0, cost_fn=1.0):
    return cost_fp * fp + cost_fn * fn

cost_fp = 1
cost_fn = 10

loss_A = weighted_error(fp=50, fn=50, cost_fp=cost_fp, cost_fn=cost_fn)
loss_B = weighted_error(fp=0, fn=100, cost_fp=cost_fp, cost_fn=cost_fn)

print("Cost(FP) =", cost_fp)
print("Cost(FN) =", cost_fn)
print("Model A weighted loss =", loss_A)
print("Model B weighted loss =", loss_B)


# 14. What a Performance Claim Must Specify

A statement such as

\[
\mathrm{Accuracy}=95\%
\]

is incomplete without the conditions under which it was obtained.

Scientific interpretation can require information about

\[
\text{metric definition},
\]

\[
\text{sample size},
\]

\[
\text{class distribution},
\]

\[
\text{sampling process},
\]

\[
\text{decision threshold},
\]

\[
\text{error structure},
\]

\[
\text{uncertainty},
\]

\[
\text{application context},
\]

and potentially

\[
\text{distribution shift}.
\]


# 15. Statistical Evidence Is Not Mathematical Proof

Mathematical proof and empirical model validation are fundamentally different.

A theorem can be established deductively under axioms and assumptions.

Model evaluation is performed using finite observations generated under particular conditions.

Therefore,

\[
\boxed{
\text{Mathematical Proof}
\neq
\text{Statistical Evidence}
\neq
\text{Empirical Observation}
}
\]

For model validation, the objective is usually to quantify evidence, uncertainty, limitations, and conditions of use rather than to prove that a model will always work.


# 16. Scientific Validation Chain

\[
\boxed{
\text{Claim}
\rightarrow
\text{Measurement}
\rightarrow
\text{Uncertainty}
\rightarrow
\text{Evidence}
\rightarrow
\text{Validation}
\rightarrow
\text{Trust Decision}
}
\]

A trust decision is not mathematical certainty. It is a decision supported by measurable evidence under stated assumptions and conditions.


# 17. OpenVals Perspective

A production AI system may require evidence across multiple dimensions, including

\[
\mathrm{Performance},
\quad
\mathrm{Reliability},
\quad
\mathrm{Robustness},
\quad
\mathrm{Safety},
\quad
\mathrm{Consistency},
\quad
\mathrm{Latency},
\quad
\mathrm{Cost},
\quad
\mathrm{Drift}.
\]

The broader validation question is

\[
\boxed{
\text{What measurable evidence supports trusting this AI system for this intended use?}
}
\]

This is the scientific direction behind OpenVals.


# 18. Episode 02 Summary

A classifier can have

\[
95\%
\]

accuracy and

\[
0\%
\]

recall on the positive class.

Equal accuracy does not imply equal error behaviour:

\[
\mathrm{Accuracy}_A
=
\mathrm{Accuracy}_B
\not\Rightarrow
\mathrm{Behaviour}_A
=
\mathrm{Behaviour}_B.
\]

Under an independent Bernoulli model,

\[
\mathrm{Var}(\hat p)
=
\frac{p(1-p)}{n}.
\]

Hence sample size affects uncertainty.

Class imbalance can make raw accuracy misleading.

Performance evaluation therefore requires both mathematical measurement and scientifically justified interpretation.

## Central conclusion

\[
\boxed{
\text{A metric is a number. Evidence requires context.}
}
\]

### Next Episode

**Episode 03 — Probability, Uncertainty, and Why AI Predictions Are Not Certainties**
